In [1]:
from pathlib import Path
import pandas as pd

In [5]:
raw_data = Path('../data/raw')
recs = pd.read_csv(raw_data/'recommendations.csv', usecols=["user_id", "app_id", "date", "is_recommended", "hours"], 
                   parse_dates=["date"])
games = pd.read_csv(raw_data / "games.csv")

print(recs.shape, games.shape)
print(recs.dtypes)
print(recs.isna().sum())
print(recs.head())

(41154794, 5) (50872, 13)
app_id                     int64
date              datetime64[ns]
is_recommended              bool
hours                    float64
user_id                    int64
dtype: object
app_id            0
date              0
is_recommended    0
hours             0
user_id           0
dtype: int64
    app_id       date  is_recommended  hours  user_id
0   975370 2022-12-12            True   36.3    51580
1   304390 2017-02-17           False   11.5     2586
2  1085660 2019-11-17            True  336.5   253880
3   703080 2022-09-23            True   27.4   259432
4   526870 2021-01-10            True    7.9    23869


In [6]:
audit = {
"rows": len(recs),
"missing_hours": int(recs["hours"].isna().sum()),
"duplicate_pairs": int(recs.duplicated(["user_id", "app_id"]).sum()),
"min_hours": float(recs["hours"].min()),
"max_hours": float(recs["hours"].max()),
}
print(audit)

{'rows': 41154794, 'missing_hours': 0, 'duplicate_pairs': 21, 'min_hours': 0.0, 'max_hours': 1000.0}


In [9]:
out = Path('../data/processed/chunks')
out.mkdir(parents=True, exist_ok=True)

cols = ["user_id", "app_id", "date", "is_recommended", "hours"]
for idx, chunk in enumerate(pd.read_csv(
"../data/raw/recommendations.csv", usecols=cols,
parse_dates=["date"], chunksize=500_000)):
    chunk = chunk.dropna(subset=["user_id", "app_id", "date"])
    chunk = chunk[chunk["hours"].ge(0) & chunk["hours"].notna()]
    chunk.to_parquet(out / f"part-{idx:04d}.parquet", index=False)